<a href="https://colab.research.google.com/github/fanat503/Induction-Heads-Tinystories/blob/main/sae.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
class SparseAutoEncoder(nn.Module):
    def __init__(self, d_model = 768, n_features = 8192):
        super().__init__()
        self.encoder = nn.Linear(d_model, n_features)
        self.decoder = nn.Linear(n_features, d_model, bias=False)
        self.pre_bias = nn.Parameter(torch.zeros(d_model))

    def forward(self, x):
        x_centered = x - self.pre_bias
        hidden = F.relu(self.encoder(x_centered))
        reconstructed = self.decoder(hidden) + self.pre_bias
        return reconstructed, hidden

    def loss(self, x, reconstructed, hidden, l1_coeff=0.01):
        mse = F.mse_loss(reconstructed, x)
        l1 = hidden.abs().mean()
        total = mse + l1_coeff * l1
        return total, mse, l1

@torch.no_grad()
def collect_activations(model, dataloader, layer_idx=6, n_batches=100):
    all_activations = []
    activations = {}

    def hook_fn(module, input, output):
        activations['layer'] = output

    handle = model.transformer.h[layer_idx].register_forward_hook(hook_fn)
    model.eval()
    for i in range(n_batches):
        x, y = dataloader.next_batch()
        x = x.to(device)
        model = model.to(device)
        model(x)
        acts = activations['layer'].detach()
        acts = acts.view(-1, acts.shape[-1])
        all_activations.append(acts)

    handle.remove()
    return torch.cat(all_activations, dim=0)

def train_sae(activations, n_epochs=5, batch_size=4096, lr=3e-4, l1_coeff=0.01):
    sae = SparseAutoEncoder(d_model=768, n_features=8192).to(device)

    optimizer = torch.optim.AdamW(sae.parameters(), lr=lr)

    dataset = torch.utils.data.TensorDataset(activations)
    loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)

    for epoch in range(n_epochs):
      all_hidden = []
      total_loss = 0
      total_mse = 0
      n_batches = 0
      for (batch,) in loader:
          batch = batch.to(device)
          reconstructed, hidden = sae(batch)
          total, mse, l1 = sae.loss(batch, reconstructed, hidden, l1_coeff)

          optimizer.zero_grad()
          total.backward()
          optimizer.step()

          all_hidden.append(hidden.detach().cpu())
          total_loss += total.item()
          total_mse += mse.item()
          n_batches += 1

      all_hidden = torch.cat(all_hidden, dim=0)

      avg_active = ((all_hidden > 0).float().sum(dim=1)).mean()
      dead_features = (all_hidden.max(dim=0).values == 0).sum()

      print(f"Epoch {epoch+1}/{n_epochs} | "
            f"Loss: {total_loss/n_batches:.4f} | "
            f"MSE: {total_mse/n_batches:.4f} | "
            f"Active: {avg_active:.1f} | "
            f"Dead: {dead_features}/8192")

    return sae

@torch.no_grad()
def find_top_activating(sae, model, dataloader, feature_id,
                         n_batches=50, top_k=10, layer_idx=6):
    model.eval()
    results = []
    activations = {}

    def hook_fn(module, input, output):
        activations['layer'] = output

    handle = model.transformer.h[layer_idx].register_forward_hook(hook_fn)

    for i in range(n_batches):
        x, y = dataloader.next_batch()
        x = x.to(device)
        B, T = x.shape

        model(x)

        acts = activations['layer'].detach()
        acts = acts.view(-1, acts.shape[-1])

        reconstructed, hidden = sae(acts)
        score = hidden[:, feature_id]

        for pos in range(len(score)):
            if score[pos] > 0:
                b_idx = pos // T
                t_idx = pos % T

                context_start = max(0, t_idx - 5)
                context_end = min(T, t_idx + 3)
                context = x[b_idx, context_start:context_end].tolist()

                results.append({
                    'score': score[pos].item(),
                    'token_pos': t_idx,
                    'context': context
                })

    handle.remove()
    results.sort(key=lambda r: r['score'], reverse=True)
    return results[:top_k]